# Silver - Pedidos Cabeçalho

Processamento e padronização do cabeçalho dos pedidos comerciais.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'erp_pedidos_cabecalho'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{table_name}'

In [ ]:
from pyspark.sql.functions import col, to_timestamp, coalesce, lit, trim

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .select(
        col("id_pedido").cast("integer").alias("id_pedido"),
        col("id_cliente").cast("integer").alias("id_cliente"),
        col("id_vendedor").cast("integer").alias("id_vendedor"),
        to_timestamp(col("data_pedido")).alias("data_pedido"),
        coalesce(trim(col("status_pedido")), lit("Desconhecido")).cast("string").alias("status_pedido")
    )
    .filter(col("id_pedido").isNotNull())
    .dropDuplicates(["id_pedido"])
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['status_pedido'],
    chave_upsert='id_pedido'
)